# Airbnb Marketplace: SQL Experiment Setup

This notebook picks up where Notebook 1 left off — taking the 
cleaned data and using SQL to dig into the selection bias 
problem, then fixing it with propensity score matching before 
exporting a dataset that's actually ready for the analysis in 
Notebook 3.

Roughly, here's what happens:
1. Load the cleaned data into SQLite
2. Explore it with SQL to confirm what's different between groups
3. Run PSM to build comparable control and treatment groups
4. Pull in the covariate I'll need for CUPED later
5. Export the matched dataset for Notebook 3

## Imports

In [36]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

## Load clean data

In [39]:
# Load clean datasets from Notebook 1
path = "data/"

listings_clean = pd.read_csv(path + "listings_clean.csv")
calendar_clean = pd.read_csv(path + "calendar_clean.csv")

print(f'Listings : {listings_clean.shape}')
print(f'Calendar : {calendar_clean.shape}')

Listings : (36613, 23)
Calendar : (13495986, 6)


## Set up SQLite database

In [31]:
# set up sqlite and load tables
conn = sqlite3.connect(path + "airbnb_experiment.db")

listings_clean.to_sql('listings', conn, if_exists='replace', index=False)
calendar_clean.to_sql('calendar', conn, if_exists='replace', index=False)

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print('tables loaded:')
print(tables)

tables loaded:
       name
0  listings
1  calendar


In [33]:
print('=== General table overview ===')
print(f'total listings: {len(listings_clean):,}')
print(f'\nfirst 5 rows:')
print(listings_clean.head())

print(f'\nsummary stats:')
print(listings_clean.describe())


=== General table overview ===
total listings: 36,613

first 5 rows:
     id  host_id  instant_bookable        room_type neighbourhood_cleansed  \
0  2539     2787             False     Private room             Kensington   
1  2595     2845             False  Entire home/apt                Midtown   
2  5136     7378              True  Entire home/apt            Sunset Park   
3  6848    15991             False  Entire home/apt           Williamsburg   
4  6872    16104             False     Private room            East Harlem   

  neighbourhood_group_cleansed  accommodates  review_scores_rating  \
0                     Brooklyn             2                4.8900   
1                    Manhattan             1                4.6800   
2                     Brooklyn             4                4.7500   
3                     Brooklyn             3                4.5800   
4                    Manhattan             1                5.0000   

   estimated_occupancy_l365d  number_of_r

## SQL exploratory analysis

Before jumping into PSM, I wanted to confirm with SQL what the 
EDA in Notebook 1 already hinted at — that Instant Book and 
Request to Book listings really do look different from each 
other, and by how much.

## Group Overview


In [28]:
group_overview = pd.read_sql("""
    SELECT
        CASE WHEN instant_bookable = 1 THEN 'Instant Book' ELSE 'Request to Book' END AS group_name,
        COUNT(*) AS listing_count,
        AVG(estimated_occupancy_l365d) AS avg_occupancy,
        AVG(price) AS avg_price,
        AVG(review_scores_rating) AS avg_review_score,
        AVG(accommodates) AS avg_accommodates
    FROM listings
    GROUP BY instant_bookable
""", conn)

print('group overview:')
print(group_overview.round(2).to_string())

group overview:
        group_name  listing_count  avg_occupancy  avg_price  avg_review_score  avg_accommodates
0  Request to Book          29108        44.0100   176.4800            3.3300            2.6800
1     Instant Book           7505        56.9900   279.6300            2.8400            3.1100


# Room Type Distribution

In [20]:
room_type = pd.read_sql(""" 
    SELECT 
        room_type,
        CASE WHEN instant_bookable = 1 THEN 'Instant Book' 
             ELSE 'Request to Book' END AS group_name,
        COUNT(*) AS Total_Number,
        ROUND(COUNT(*) * 100.0 / 
              SUM(COUNT(*)) OVER (PARTITION BY instant_bookable), 2) AS percentage
    FROM listings
    GROUP BY instant_bookable, room_type
    ORDER BY instant_bookable, total_number desc ,room_type
     """,conn)
print(room_type)

         room_type       group_name  Total_Number  percentage
0  Entire home/apt  Request to Book         15650     53.7700
1     Private room  Request to Book         13311     45.7300
2      Shared room  Request to Book           135      0.4600
3       Hotel room  Request to Book            12      0.0400
4  Entire home/apt     Instant Book          4056     54.0400
5     Private room     Instant Book          2926     38.9900
6       Hotel room     Instant Book           479      6.3800
7      Shared room     Instant Book            44      0.5900


# Neighborhood Distribution

In [21]:
borough_dist = pd.read_sql("""
    SELECT 
        neighbourhood_group_cleansed AS borough,
        CASE WHEN instant_bookable = 1 
             THEN 'Instant Book' 
             ELSE 'Request to Book' 
        END AS group_name,
        COUNT(*) AS listings,
        COUNT(*) * 100.0 / 
            SUM(COUNT(*)) OVER (PARTITION BY instant_bookable) AS percentage
    FROM listings
    GROUP BY neighbourhood_group_cleansed, instant_bookable
    ORDER BY borough, instant_bookable
""", conn)
print('Borough distribution:')
print(borough_dist.round(2).to_string())
    

Borough distribution:
         borough       group_name  listings  percentage
0          Bronx  Request to Book       961      3.3000
1          Bronx     Instant Book       226      3.0100
2       Brooklyn  Request to Book     11495     39.4900
3       Brooklyn     Instant Book      1937     25.8100
4      Manhattan  Request to Book     11862     40.7500
5      Manhattan     Instant Book      4494     59.8800
6         Queens  Request to Book      4471     15.3600
7         Queens     Instant Book       805     10.7300
8  Staten Island  Request to Book       319      1.1000
9  Staten Island     Instant Book        43      0.5700


In [22]:
neighbourhood_dist = pd.read_sql("""
    SELECT 
        neighbourhood_cleansed AS neighbourhood,
        neighbourhood_group_cleansed AS borough,
        CASE WHEN instant_bookable = 1 
             THEN 'Instant Book' 
             ELSE 'Request to Book' 
        END AS group_name,
        COUNT(*) AS count,
        COUNT(*) * 100.0 / 
            SUM(COUNT(*)) OVER (PARTITION BY instant_bookable) AS pct
    FROM listings
    GROUP BY neighbourhood_cleansed, 
             neighbourhood_group_cleansed, 
             instant_bookable
    ORDER BY borough, neighbourhood, instant_bookable
""", conn)
print('\nNeighbourhood distribution (top 20 by count):')
print(neighbourhood_dist.nlargest(20, 'count').to_string())


Neighbourhood distribution (top 20 by count):
          neighbourhood    borough       group_name  count     pct
97   Bedford-Stuyvesant   Brooklyn  Request to Book   2246  7.7161
182        Williamsburg   Brooklyn  Request to Book   1813  6.2285
206              Harlem  Manhattan  Request to Book   1446  4.9677
113            Bushwick   Brooklyn  Request to Book   1269  4.3596
242     Upper East Side  Manhattan  Request to Book   1165  4.0023
208      Hell's Kitchen  Manhattan  Request to Book   1124  3.8615
244     Upper West Side  Manhattan  Request to Book   1097  3.7687
220             Midtown  Manhattan  Request to Book   1068  3.6691
127       Crown Heights   Brooklyn  Request to Book    988  3.3943
221             Midtown  Manhattan     Instant Book    965 12.8581
196        East Village  Manhattan  Request to Book    802  2.7553
188             Chelsea  Manhattan  Request to Book    606  2.0819
194         East Harlem  Manhattan  Request to Book    581  1.9960
154          Gr

In [9]:
# Check if specific neighbourhoods are disproportionate
nb_pivot = pd.read_sql("""
    SELECT
        neighbourhood_cleansed,
        neighbourhood_group_cleansed AS borough,
        SUM(CASE WHEN instant_bookable = 1 THEN 1 ELSE 0 END) AS instant_book,
        SUM(CASE WHEN instant_bookable = 0 THEN 1 ELSE 0 END) AS request_to_book,
        ROUND(
            SUM(CASE WHEN instant_bookable = 1 THEN 1.0 ELSE 0 END) /
            COUNT(*) * 100, 1
        ) AS ib_pct
    FROM listings
    GROUP BY neighbourhood_cleansed, neighbourhood_group_cleansed
    HAVING COUNT(*) > 50
    ORDER BY ib_pct DESC
    LIMIT 20
""", conn)
print('Top 20 neighbourhoods by Instant Book concentration:')
print(nb_pivot.to_string())

Top 20 neighbourhoods by Instant Book concentration:
   neighbourhood_cleansed    borough  instant_book  request_to_book  ib_pct
0        Theater District  Manhattan           288              111 72.2000
1           Fort Hamilton   Brooklyn            62               37 62.6000
2                 Midtown  Manhattan           965             1068 47.5000
3             Murray Hill  Manhattan           208              233 47.2000
4                 Tribeca  Manhattan            70               93 42.9000
5      Financial District  Manhattan           258              350 42.4000
6       Flatiron District  Manhattan            29               52 35.8000
7       Battery Park City  Manhattan            30               55 35.3000
8       Downtown Brooklyn   Brooklyn            25               50 33.3000
9                    SoHo  Manhattan            84              182 31.6000
10                Chelsea  Manhattan           264              606 30.3000
11           West Village  Manhatta

# Price Tier Distribution

In [23]:
# Question: are Instant Book listings concentrated in a price tier?
price_tier = pd.read_sql("""
    SELECT
        CASE WHEN instant_bookable = 1 THEN 'Instant Book' 
             ELSE 'Request to Book' END AS group_name,
        CASE 
            WHEN price < 100 THEN '1_budget (<$100)'
            WHEN price BETWEEN 100 AND 250 THEN '2_mid ($100-250)'
            ELSE '3_luxury (>$250)'
        END AS price_tier,
        COUNT(*) AS count,
        ROUND(COUNT(*) * 100.0 / 
              SUM(COUNT(*)) OVER (PARTITION BY instant_bookable), 2) AS pct
    FROM listings
    GROUP BY instant_bookable, price_tier
    ORDER BY group_name, price_tier
""", conn)
print('\nPrice tier distribution:')
print(price_tier.to_string())


Price tier distribution:
        group_name        price_tier  count     pct
0     Instant Book  1_budget (<$100)   2006 26.7300
1     Instant Book  2_mid ($100-250)   2676 35.6600
2     Instant Book  3_luxury (>$250)   2823 37.6100
3  Request to Book  1_budget (<$100)  11959 41.0800
4  Request to Book  2_mid ($100-250)  14059 48.3000
5  Request to Book  3_luxury (>$250)   3090 10.6200


# Accomadates Distribution

In [24]:
# Summary stats for accommodates by group
accommodates_summary = pd.read_sql("""
    SELECT
        CASE WHEN instant_bookable = 1 
             THEN 'Instant Book' 
             ELSE 'Request to Book' 
        END AS group_name,
        MIN(accommodates)   AS min_acc,
        MAX(accommodates)   AS max_acc,
        AVG(accommodates)   AS avg_acc,
        COUNT(*) * 100.0 /
            SUM(COUNT(*)) OVER (PARTITION BY instant_bookable) AS pct
    FROM listings
    GROUP BY instant_bookable
""", conn)
print('\nAccommodates summary:')
print(accommodates_summary.round(2).to_string())


Accommodates summary:
        group_name  min_acc  max_acc  avg_acc      pct
0  Request to Book        1       16   2.6800 100.0000
1     Instant Book        1       16   3.1100 100.0000


## Propensity Score Matching (PSM)

### Why I needed this

The SQL exploration confirmed what the EDA suggested — Instant 
Book and Request to Book listings aren't really comparable as-is. 
They differ systematically on price, room type, review scores, 
and accommodates. If I just compared the two groups directly, 
any difference I found could just be explained by those 
pre-existing gaps, not by Instant Book itself.

So I used PSM to fix that:
1. Estimate each listing's probability of being Instant Book, 
   based on its characteristics — this is the propensity score
2. Match each Instant Book listing to the closest non-Instant 
   Book listing by that score
3. Check that the matched groups actually ended up balanced 
   before doing any analysis

### What I matched on

- `price` — nightly price
- `room_type` — entire home vs private room vs shared room
- `review_scores_rating` — overall listing quality
- `accommodates` — how many guests it fits
- `neighbourhood_group_cleansed` — which borough

In [34]:
# prep features for the propensity model — one hot encode the categoricals
psm_data = pd.get_dummies(
    listings_clean[['id', 'instant_bookable',
                    'estimated_occupancy_l365d',
                    'price', 'review_scores_rating',
                    'accommodates', 'minimum_nights',
                    'room_type', 'neighbourhood_cleansed',
                    'neighbourhood_group_cleansed']],
    columns=['room_type', 'neighbourhood_cleansed',
             'neighbourhood_group_cleansed'],
    drop_first=True
).fillna(0)

feature_cols = [c for c in psm_data.columns
                if c not in ['id', 'instant_bookable', 'estimated_occupancy_l365d']]

X = psm_data[feature_cols].values
y = psm_data['instant_bookable'].astype(int).values

# scale before logistic regression so price doesn't dominate everything
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# fit logistic regression to get propensity scores
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_scaled, y)
psm_data['propensity_score'] = lr.predict_proba(X_scaled)[:, 1]

print(f'IB score mean  : {psm_data[psm_data.instant_bookable==1].propensity_score.mean():.3f}')
print(f'RTB score mean : {psm_data[psm_data.instant_bookable==0].propensity_score.mean():.3f}')

# now match each instant book listing to its closest non-instant book listing
# using a 0.05 caliper so we don't force bad matches
CALIPER = 0.05
treatment = psm_data[psm_data['instant_bookable'] == 1].copy()
control   = psm_data[psm_data['instant_bookable'] == 0].copy()

matched_pairs = []
used_control_ids = set()

for _, treat_row in treatment.iterrows():
    available = control[~control['id'].isin(used_control_ids)]
    if len(available) == 0:
        break

    distances = abs(available['propensity_score'] - treat_row['propensity_score'])

    # skip if nothing close enough
    if distances.min() > CALIPER:
        continue

    match = available.loc[distances.idxmin()]
    matched_pairs.append({
        'treatment_id': treat_row['id'],
        'control_id': match['id'],
        'distance': distances.min()
    })
    used_control_ids.add(match['id'])

matched_pairs_df = pd.DataFrame(matched_pairs)
print(f'\nMatched pairs             : {len(matched_pairs_df):,}')
print(f'Avg score distance        : {matched_pairs_df["distance"].mean():.4f}')
print(f'Treatment lost to caliper : {len(treatment) - len(matched_pairs_df):,}')

# pull the full listing rows for matched ids and tag with experiment group
treat_matched = listings_clean[listings_clean['id'].isin(matched_pairs_df['treatment_id'])].copy()
ctrl_matched  = listings_clean[listings_clean['id'].isin(matched_pairs_df['control_id'])].copy()

treat_matched['experiment_group'] = 'treatment'
ctrl_matched['experiment_group']  = 'control'

matched_df = pd.concat([treat_matched, ctrl_matched], ignore_index=True)

print(f'\nFinal matched dataset:')
print(f'Treatment : {len(treat_matched):,}')
print(f'Control   : {len(ctrl_matched):,}')
print(f'Total     : {len(matched_df):,}')

# balance check — did matching actually fix the confounders?
print('\n' + '='*50)
print('Balance check — before vs after PSM')
print('='*50)

for var in ['price', 'review_scores_rating', 'accommodates', 'minimum_nights']:
    before = abs(
        listings_clean[listings_clean.instant_bookable==True][var].mean() -
        listings_clean[listings_clean.instant_bookable==False][var].mean()
    )
    after = abs(treat_matched[var].mean() - ctrl_matched[var].mean())
    flag = 'improved' if after < before else 'worse'
    print(f'{var:25s} before: {before:.2f} -> after: {after:.2f} ({flag})')

# same check for neighbourhood — Midtown was a big confounder in EDA
nb_before = pd.crosstab(listings_clean['neighbourhood_cleansed'],
                         listings_clean['instant_bookable'], normalize='columns')
nb_after = pd.crosstab(matched_df['neighbourhood_cleansed'],
                        matched_df['experiment_group'], normalize='columns')

before_diff = abs(nb_before[True] - nb_before[False]).mean()
after_diff  = abs(nb_after['treatment'] - nb_after['control']).mean()

print(f'\nNeighbourhood balance:')
print(f'Avg absolute diff before PSM : {before_diff:.4f}')
print(f'Avg absolute diff after PSM  : {after_diff:.4f}')
print(f'Improvement: {after_diff < before_diff}')

# convert to a 0-1 rate so it lines up with the power analysis (which used a rate)
matched_df['occupancy_rate'] = matched_df['estimated_occupancy_l365d'] / 365

print(f'\nOccupancy rate by group:')
print(matched_df.groupby('experiment_group')['occupancy_rate'].agg(['mean','std','count']).round(4))

# save for notebook 3
matched_df.to_csv(path + "matched_listings.csv", index=False)
print(f'\nExported: {matched_df.shape}')
print(f'Columns: {matched_df.columns.tolist()}')

IB score mean  : 0.312
RTB score mean : 0.177

Matched pairs             : 6,765
Avg score distance        : 0.0010
Treatment lost to caliper : 740

Final matched dataset:
Treatment : 6,765
Control   : 6,765
Total     : 13,530

Balance check — before vs after PSM
price                     before: 103.15 -> after: 5.90 (improved)
review_scores_rating      before: 0.49 -> after: 0.10 (improved)
accommodates              before: 0.43 -> after: 0.15 (improved)
minimum_nights            before: 6.61 -> after: 2.93 (improved)

Neighbourhood balance:
Avg absolute diff before PSM : 0.0023
Avg absolute diff after PSM  : 0.0007
Improvement: True

Occupancy rate by group:
                   mean    std  count
experiment_group                     
control          0.1205 0.2266   6765
treatment        0.1643 0.2621   6765

Exported: (13530, 25)
Columns: ['id', 'host_id', 'instant_bookable', 'room_type', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'accommodates', 'review_scores_rating

### Notebook 2 summary

PSM matched 6,765 pairs — comfortably above the 4,662 I 
needed from the power analysis, about a 45% buffer. Every 
confounder I checked got noticeably closer between groups 
after matching:

- Price gap went from 103 USD down to 5.90 USD (94% closer)
- Review score gap went from 0.49 to 0.10 (80% closer)
- Accommodates gap went from 0.43 to 0.15 (65% closer)
- Minimum nights gap went from 6.61 to 2.93 (56% closer)
- Neighbourhood balance improved by about 70%

The raw occupancy difference between groups came out to 4.38 
points (treatment higher), already past my 2.0 point MDE. 
CUPED and the actual hypothesis test in Notebook 3 will tell 
me whether that holds up statistically.

### PSM limitation — what matching can't fix

Even though PSM closed most of the gap on the things I could 
measure (price went from a $103 difference down to $5.90, 
roughly 4% of the average price), it can only balance what's 
actually in the dataset. Things like photo quality, how 
responsive a host is, or how well-written a listing description 
is aren't in here at all — and those could easily differ 
between Instant Book and non-Instant Book listings in ways 
that affect occupancy too. That's the main reason I'm treating 
this as an association rather than a causal claim.

### Scope limitation — listings I had to leave out

About 740 Instant Book listings (roughly 10%) didn't get 
matched at all, because there was no comparable control 
listing close enough within my caliper. These are probably 
the more unusual ones — extreme prices, unique locations, 
that sort of thing. So what I'm reporting applies to the 
"typical" Instant Book listing that has a real counterpart 
out there, not necessarily the full population of Instant 
Book listings.

In [38]:
# save for notebook 3
matched_df.to_csv(path + "matched_listings.csv", index=False)
print(f'\nExported: {matched_df.shape}')
print(f'Columns: {matched_df.columns.tolist()}')


Exported: (13530, 25)
Columns: ['id', 'host_id', 'instant_bookable', 'room_type', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'accommodates', 'review_scores_rating', 'estimated_occupancy_l365d', 'number_of_reviews_ltm', 'number_of_reviews', 'number_of_reviews_ly', 'price', 'property_type', 'minimum_nights', 'host_is_superhost', 'host_identity_verified', 'availability_30', 'availability_365', 'latitude', 'longitude', 'has_reviews', 'ib_label', 'experiment_group', 'occupancy_rate']
